# CReST — Source-only summary features + Ridge baseline

**Purpose:** Compare a deliberately simple, feature-based Ridge regression with the existing CReST `z_general + Ridge` held-out-item prediction experiment. This notebook follows the uploaded **01_Data_Preparation_and_Design_Audit** handoff structure and **06_CReST_Final_Evaluation**, Cells 20–23. It does not retrain or alter CReST.

**Exact shared conditions:** same 32 items, A/B sets, 7 country-fold partitions, validation-system-only supervised Ridge fitting, locked held-out education systems, eligibility, target (`mean normalized credit`), and system-equal Spearman correlation. The baseline uses 36 hand-crafted source-only indicators derived from observed credit, behaviour and log-process channels (3 ideation processes × [7 summaries + 5 availability flags]). No student or school background variables, booklet identifiers, or official PISA plausible values enter its features.

**Important:** This is an **additional post-hoc comparison**, not a pre-specified confirmatory test and not an architecture ablation. It isolates the benefit of learned vs. manually aggregated representations under the frozen-probe protocol, but feature granularity/calibration differs. Report all outcomes regardless of which model performs better.

**Status:** The uploaded two notebooks contain code but not the exported PISA arrays and existing CReST prediction `.npz` files; final numerical results cannot be computed here. Run this notebook in the *existing project root* after notebook 01 and 06 have created their required handoff/evaluation outputs.

In [1]:
from pathlib import Path
import os, json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

# Change this only if the notebook is not in your project root.
BASE_DIR = Path(os.environ.get("CREST_PROJECT_DIR", str(Path.cwd()))).expanduser().resolve()
HANDOFF_DIR = BASE_DIR / "processed" / "crest_v5_handoff"
FINAL_TEST_ROOT = BASE_DIR / "processed" / "crest_final_test_sets"
ORIGINAL_PRED_DIR = BASE_DIR / "results" / "creative_signal_safeguard"
BASELINE_DIR = BASE_DIR / "results" / "ridge_summary_baseline"
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

ITEM_SET_A = np.array([0,1,5,6,7,8,9,10,12,15,17,20,24,27,28,31], dtype=np.int64)
ITEM_SET_B = np.array([2,3,4,11,13,14,16,18,19,21,22,23,25,26,29,30], dtype=np.int64)
DIRECTIONS = {"A_to_B": (ITEM_SET_A, ITEM_SET_B), "B_to_A": (ITEM_SET_B, ITEM_SET_A)}
ALPHA = 1.0             # Exactly as in Notebook 06; no hyperparameter search
BOOTSTRAP_SEED = 20261014
N_BOOTSTRAP = 10_000
assert np.array_equal(np.sort(np.r_[ITEM_SET_A,ITEM_SET_B]), np.arange(32))
print("Project root:", BASE_DIR)

Project root: D:\Kubra (Umit blg)\Kubra (papers)\PISA-CReST\Dataset and codes


In [2]:
# Strict input check; don't quietly reconstruct different cohorts.
required = [
    HANDOFF_DIR / "primary_meta.pkl",
    HANDOFF_DIR / "handoff_config.json",
    *[HANDOFF_DIR / (n + ".npy") for n in (
        "credit_tokens", "behaviour_tokens", "performance_mask",
        "interaction_mask", "process_tensor", "item_ideation_ids"
    )],
    *[FINAL_TEST_ROOT / f"fold_{k}" / "test_indices.npy" for k in range(7)]
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Required outputs from Notebook 01/06 not found. Run in the original project "
        "directory, with the existing data exports. Missing:\n" + "\n".join(missing)
    )

with open(HANDOFF_DIR / "handoff_config.json", encoding="utf-8") as f:
    cfg = json.load(f)
primary_meta = pd.read_pickle(HANDOFF_DIR / "primary_meta.pkl").reset_index(drop=True)
credit = np.load(HANDOFF_DIR / "credit_tokens.npy", mmap_mode="r")
behaviour = np.load(HANDOFF_DIR / "behaviour_tokens.npy", mmap_mode="r")
performance_mask = np.load(HANDOFF_DIR / "performance_mask.npy", mmap_mode="r")
interaction_mask = np.load(HANDOFF_DIR / "interaction_mask.npy", mmap_mode="r")
process = np.load(HANDOFF_DIR / "process_tensor.npy", mmap_mode="r")
ideation = np.load(HANDOFF_DIR / "item_ideation_ids.npy", mmap_mode="r")

N = len(primary_meta)
assert N == int(cfg["n_students"]) == 142_564, "Check original analysis cohort/version"
assert primary_meta["CNT"].nunique() == 63
assert set(primary_meta["country_fold"]) == set(range(7))
assert credit.shape == behaviour.shape == performance_mask.shape == interaction_mask.shape == (N,32)
assert process.shape == (N,32,3) and ideation.shape == (32,)
assert sorted(np.unique(ideation).tolist()) == [0,1,2]
assert cfg.get("process_order") == ["GDI", "GCI", "EII"]
assert np.isin(np.unique(credit), [0,1,2,3]).all()
assert np.isin(np.unique(behaviour), [0,1,2,3]).all()
assert primary_meta.groupby("CNT")["country_fold"].nunique().eq(1).all()

fold_values = primary_meta["country_fold"].to_numpy(dtype=np.int64)
country_values = primary_meta["CNT"].to_numpy()
print(f"Validated data: {N:,} students, 63 systems, 32 items.")

Validated data: 142,564 students, 63 systems, 32 items.


## Feature construction (source items only)

For each ideation process **GDI/GCI/EII**, compute: (a) mean normalized scorable credit; (b) three response-behaviour proportions (scorable, no response, not reached); and (c) three mean available log-process channels. Add one presence flag for credit, one for behaviour, and three for the process channels. Thus `3 × (1+3+3+1+1+3) = 36` columns.

A missing group summary is filled with zero **and its availability is explicitly flagged**; no target-item masks, credit or process values are inspected. Raw `process_tensor` already contains the log1p transformation from Notebook 01; no target data, whole-data imputation or fit-dependent transforms are used to generate features. `StandardScaler` is fitted only within the corresponding validation systems through the Ridge pipeline.

In [3]:
def masked_mean(values, valid):
    valid = np.asarray(valid, dtype=bool)
    values = np.asarray(values, dtype=np.float64)
    total = np.where(valid, values, 0.0).sum(axis=1)
    count = valid.sum(axis=1)
    return np.divide(total, count, out=np.zeros(len(total), dtype=np.float64), where=count>0), (count>0).astype(np.float64)


def source_summary_features(indices, source_items, *, arrays=None):
    """Returns (n, 36) features; only supplied `source_items` are read."""
    if arrays is None:
        cc, bb, pm, im, pr, ip = credit, behaviour, performance_mask, interaction_mask, process, ideation
    else:
        cc, bb, pm, im, pr, ip = arrays
    # np.ix_ selects only SOURCE items before any computation.
    ct = np.asarray(cc[np.ix_(indices, source_items)])
    bt = np.asarray(bb[np.ix_(indices, source_items)])
    m_credit = np.asarray(pm[np.ix_(indices, source_items)], dtype=bool)
    m_interact = np.asarray(im[np.ix_(indices, source_items)], dtype=bool)
    px = np.asarray(pr[indices][:, source_items, :], dtype=np.float64)
    proc_of_item = np.asarray(ip[source_items])
    features, names = [], []
    for group_id, group_name in enumerate(["GDI","GCI","EII"]):
        keep = (proc_of_item == group_id)
        if keep.sum() == 0:
            raise ValueError(f"No source items from {group_name}")
        c = ct[:, keep]
        b = bt[:, keep]
        mc = m_credit[:, keep] & np.isin(c, [1,2,3])
        mi = m_interact[:, keep] & np.isin(b, [1,2,3])
        p = px[:, keep, :]
        values = []
        mean_credit, credit_present = masked_mean((c.astype(float)-1.0)/2.0, mc)
        values.append(("mean_credit", mean_credit))
        count_b = mi.sum(axis=1)
        for state, name in [(1,"scorable_rate"),(2,"no_response_rate"),(3,"not_reached_rate")]:
            numerator = ((b==state)&mi).sum(axis=1)
            rate = np.divide(numerator, count_b, out=np.zeros(len(indices),dtype=float), where=count_b>0)
            values.append((name, rate))
        process_avail = []
        for k, name in enumerate(["log_total_time","log_first_action","log_actions"]):
            valid = np.isfinite(p[:,:,k]) & mi
            avg, present = masked_mean(p[:,:,k], valid)
            values.append((name, avg))
            process_avail.append(("available_"+name, present))
        values.append(("credit_available", credit_present))
        values.append(("behaviour_available", (count_b>0).astype(float)))
        values.extend(process_avail)
        for name, value in values:
            features.append(np.asarray(value,dtype=float))
            names.append(f"{group_name}_{name}")
    X = np.column_stack(features).astype(np.float32)
    assert X.shape == (len(indices),36) and np.isfinite(X).all()
    return X, names


def hidden_target(indices, target_items, *, arrays=None):
    # Same rule as observed_credit_target() in Notebook 06, Cell 21.
    cc, pm = (credit, performance_mask) if arrays is None else (arrays[0], arrays[2])
    ct = np.asarray(cc[np.ix_(indices,target_items)])
    ok = np.asarray(pm[np.ix_(indices,target_items)],dtype=bool) & np.isin(ct,[1,2,3])
    values = np.where(ok,(ct.astype(np.float32)-1)/2.0,0.0)
    counts = ok.sum(axis=1)
    mean = np.divide(values.sum(axis=1),counts,out=np.full(len(indices),np.nan,dtype=np.float32),where=counts>0)
    return mean, counts


def source_interaction_count(indices, source_items, *, arrays=None):
    arr = interaction_mask if arrays is None else arrays[3]
    return np.asarray(arr[np.ix_(indices,source_items)],dtype=bool).sum(axis=1)


def ridge():
    return Pipeline([("scale",StandardScaler()),("ridge",Ridge(alpha=ALPHA))])


def metrics(y, pred):
    if len(y)<3:
        return {"r2":np.nan,"mae":np.nan,"rho":np.nan}
    rho = float(spearmanr(y,pred).statistic) if (np.std(y)>0 and np.std(pred)>0) else np.nan
    return {"r2":float(r2_score(y,pred)),"mae":float(mean_absolute_error(y,pred)),"rho":rho}

## Leakage unit test (synthetic only)

This cell needs **no PISA data**. We deliberately corrupt all target-item values, masks and process signals; source-only features must remain bit-for-bit identical. It additionally verifies the target definition reacts to target-credit changes. This is a code-level check, not an empirical evaluation.

In [4]:
rng = np.random.default_rng(20261014)
n = 12
cc = rng.integers(1,4,size=(n,32),dtype=np.int8)
bb = np.ones((n,32),dtype=np.int8)
pm = np.ones((n,32),dtype=bool)
im = np.ones((n,32),dtype=bool)
pr = rng.uniform(0,4,size=(n,32,3)).astype(np.float32)
ip = np.arange(32)%3
sim = (cc,bb,pm,im,pr,ip)
rows = np.arange(n)
x_before,_ = source_summary_features(rows, ITEM_SET_A, arrays=sim)
y_before,_ = hidden_target(rows, ITEM_SET_B, arrays=sim)
cc2,bb2,pm2,im2,pr2,ip2 = [a.copy() for a in sim]
cc2[:,ITEM_SET_B] = 1  # target scores now all zero
bb2[:,ITEM_SET_B] = 3
pm2[:,ITEM_SET_B] = True
im2[:,ITEM_SET_B] = False
pr2[:,ITEM_SET_B,:] = 9999
x_after,_ = source_summary_features(rows, ITEM_SET_A, arrays=(cc2,bb2,pm2,im2,pr2,ip2))
y_after,_ = hidden_target(rows, ITEM_SET_B, arrays=(cc2,bb2,pm2,im2,pr2,ip2))
assert np.array_equal(x_before,x_after), "LEAKAGE: feature construction reads target items"
assert np.any(y_before!=y_after), "Sanity check: target corruption did not change y"
print("Synthetic target-leakage invariance: PASS")

Synthetic target-leakage invariance: PASS


## Locked 7-fold, 2-direction evaluation

The source/target eligibility is matched **exactly** to Notebook 06: at least one source `interaction_mask=True` and at least one scorable hidden target item. Validation labels are used only to fit `StandardScaler + Ridge(alpha=1)`. Saved TEST indices from Notebook 06 are mandatory and audited against `country_fold`. The existing CReST arrays and checkpoints are neither loaded nor modified.

In [5]:
fold_results, country_results = [], []
seen_test_rows = []
feature_names = None

for outer_fold in range(7):
    val_fold = (outer_fold + 1) % 7
    val_idx = np.flatnonzero(fold_values == val_fold)
    test_idx = np.load(FINAL_TEST_ROOT / f"fold_{outer_fold}" / "test_indices.npy")
    assert len(np.unique(test_idx)) == len(test_idx)
    assert np.all(fold_values[test_idx] == outer_fold)
    assert np.array_equal(np.sort(test_idx),np.flatnonzero(fold_values==outer_fold))
    assert np.intersect1d(val_idx,test_idx).size == 0
    seen_test_rows.extend(test_idx.tolist())

    for direction, (source_items,target_items) in DIRECTIONS.items():
        y_val, count_val = hidden_target(val_idx,target_items)
        y_test,count_test = hidden_target(test_idx,target_items)
        ok_val = np.isfinite(y_val)&(count_val>=1)&(source_interaction_count(val_idx,source_items)>=1)
        ok_test= np.isfinite(y_test)&(count_test>=1)&(source_interaction_count(test_idx,source_items)>=1)
        assert ok_val.sum()>50 and ok_test.sum()>50
        # Extract only eligible pupils to avoid unnecessary memory use.
        val_use, test_use = val_idx[ok_val],test_idx[ok_test]
        X_val,names=source_summary_features(val_use,source_items)
        X_test,names2=source_summary_features(test_use,source_items)
        assert names==names2
        if feature_names is None: feature_names=names
        assert names==feature_names
        model=ridge()
        model.fit(X_val,y_val[ok_val])
        pred=model.predict(X_test)
        truth=y_test[ok_test]
        sys=country_values[test_use]
        pooled=metrics(truth,pred)
        system_rows=[]
        for country in sorted(np.unique(sys)):
            mask=sys==country
            m=metrics(truth[mask],pred[mask])
            row={"outer_fold":outer_fold,"direction":direction,"country":country,"n_students":int(mask.sum()),**m}
            country_results.append(row);system_rows.append(row)
        equal=pd.DataFrame(system_rows)[["r2","mae","rho"]].mean().to_dict()
        fold_results.append({"outer_fold":outer_fold,"direction":direction,
             "n_validation_students":int(ok_val.sum()),"n_test_students":int(ok_test.sum()),
             "pooled_r2":pooled["r2"],"pooled_mae":pooled["mae"],"pooled_rho":pooled["rho"],
             **{"country_equal_"+k:v for k,v in equal.items()}})
        np.savez_compressed(BASELINE_DIR/f"fold_{outer_fold}_{direction}_predictions.npz",
            handoff_row_index=test_use, target=truth.astype(np.float32),prediction=pred.astype(np.float32),
            target_item_count=count_test[ok_test].astype(np.int8))
        print(f"fold={outer_fold}  {direction}: N={len(test_use):,}, pooled rho={pooled['rho']:.4f}, system-equal rho={equal['rho']:.4f}")

assert len(seen_test_rows)==N and len(set(seen_test_rows))==N, "TEST partition coverage invalid"
fold_df=pd.DataFrame(fold_results)
country_df=pd.DataFrame(country_results)
assert len(fold_df)==14 and len(country_df)==126
assert country_df.groupby(["country","direction"]).size().eq(1).all()
fold_df.to_csv(BASELINE_DIR/"ridge_baseline_7fold_results.csv",index=False)
country_df.to_csv(BASELINE_DIR/"ridge_baseline_63system_direction_results.csv",index=False)
pd.DataFrame({"feature":feature_names}).to_csv(BASELINE_DIR/"ridge_baseline_feature_registry.csv",index=False)
print("\nSystem-equal direction means:")
print(country_df.groupby("direction")[["rho","r2","mae"]].mean().round(5))
print("Overall 63-system, two-direction mean rho:",round(country_df["rho"].mean(),5))

fold=0  A_to_B: N=21,537, pooled rho=0.5598, system-equal rho=0.5127
fold=0  B_to_A: N=21,541, pooled rho=0.5431, system-equal rho=0.4986
fold=1  A_to_B: N=20,694, pooled rho=0.6662, system-equal rho=0.5035
fold=1  B_to_A: N=20,715, pooled rho=0.6662, system-equal rho=0.5110
fold=2  A_to_B: N=20,611, pooled rho=0.5728, system-equal rho=0.5052
fold=2  B_to_A: N=20,639, pooled rho=0.5650, system-equal rho=0.5025
fold=3  A_to_B: N=20,217, pooled rho=0.5264, system-equal rho=0.4834
fold=3  B_to_A: N=20,255, pooled rho=0.4944, system-equal rho=0.4679
fold=4  A_to_B: N=19,355, pooled rho=0.6269, system-equal rho=0.5156
fold=4  B_to_A: N=19,352, pooled rho=0.6389, system-equal rho=0.5222
fold=5  A_to_B: N=19,530, pooled rho=0.6255, system-equal rho=0.4999
fold=5  B_to_A: N=19,533, pooled rho=0.6131, system-equal rho=0.4960
fold=6  A_to_B: N=19,573, pooled rho=0.5775, system-equal rho=0.5123
fold=6  B_to_A: N=19,618, pooled rho=0.5712, system-equal rho=0.5089

System-equal direction means:
   

## Paired comparison with original CReST predictions (if available)

The original Notebook 06 writes `results/creative_signal_safeguard/fold_{fold}_{direction}_test_predictions.npz`, containing handoff indices, target values and predictions. Here they are aligned by **student row index**, not by file order. Strict assertions check equal pupils, equal hidden targets and matching eligible samples. If the original prediction files are absent, this section reports what is missing and skips comparison rather than inventing CReST metrics.

In [6]:
expected_original=[ORIGINAL_PRED_DIR/f"fold_{fold}_{d}_test_predictions.npz" for fold in range(7) for d in DIRECTIONS]
missing_original=[p for p in expected_original if not p.exists()]
if missing_original:
    print("Original CReST student-level predictions unavailable; baseline results are saved.")
    print("For exact paired comparison, run Notebook 06 Cell 22 and ensure these files exist.")
    print("Missing (first 3):",*[str(p) for p in missing_original[:3]],sep="\n")
    comparison_country_df=None
else:
    old_rows=[]
    for fold in range(7):
        for d in DIRECTIONS:
            a=np.load(BASELINE_DIR/f"fold_{fold}_{d}_predictions.npz")
            b=np.load(ORIGINAL_PRED_DIR/f"fold_{fold}_{d}_test_predictions.npz")
            ia=np.asarray(a["handoff_row_index"],dtype=int)
            ib=np.asarray(b["handoff_row_index"],dtype=int)
            assert ia.size==ib.size and np.array_equal(np.sort(ia),np.sort(ib)), f"Different eligible cohorts {fold}/{d}"
            ord_a=np.argsort(ia);ord_b=np.argsort(ib)
            assert np.allclose(a["target"][ord_a],b["target"][ord_b],atol=1e-6,rtol=0),f"Targets differ {fold}/{d}"
            for idx, yt, baseline_pred, crest_pred in zip(ia[ord_a], a["target"][ord_a],a["prediction"][ord_a],b["prediction"][ord_b]):
                # Store only what is needed for paired system evaluation.
                old_rows.append((fold,d,country_values[idx],float(yt),float(baseline_pred),float(crest_pred)))
    matched=pd.DataFrame(old_rows,columns=["fold","direction","country","target","ridge_pred","crest_pred"])
    paired=[]
    for (fold,d,country), df in matched.groupby(["fold","direction","country"],sort=False):
        r=metrics(df["target"],df["ridge_pred"])["rho"]
        c=metrics(df["target"],df["crest_pred"])["rho"]
        paired.append({"fold":fold,"direction":d,"country":country,"n_students":len(df),"ridge_rho":r,"crest_rho":c,"delta_crest_minus_ridge":c-r})
    comparison_country_df=pd.DataFrame(paired)
    comparison_country_df.to_csv(BASELINE_DIR/"crest_vs_ridge_paired_by_country_direction.csv",index=False)
    print("CReST and Ridge matched on exact row IDs and hidden targets.")
    print(comparison_country_df.groupby("direction")[["ridge_rho","crest_rho","delta_crest_minus_ridge"]].mean().round(5))
    del old_rows, matched

CReST and Ridge matched on exact row IDs and hidden targets.
           ridge_rho  crest_rho  delta_crest_minus_ridge
direction                                               
A_to_B       0.53426    0.50463                 -0.02964
B_to_A       0.53061    0.52529                 -0.00532


## System-cluster bootstrap (63 paired education systems)

First average A→B and B→A **within each system**, then resample the 63 *paired system means* with replacement (10,000 replicates). This captures the manuscript's system-equal target and preserves direction pairing. The comparative interval is descriptive/post-hoc, and the original confirmatory protocol remains unchanged.

In [7]:
def bootstrap_system_mean(values,seed=BOOTSTRAP_SEED,n_rep=N_BOOTSTRAP):
    arr=np.asarray(values,dtype=float)
    assert np.isfinite(arr).all()
    rng=np.random.default_rng(seed)
    reps=np.empty(n_rep,dtype=float)
    for k in range(n_rep):
        reps[k]=arr[rng.integers(0,len(arr),len(arr))].mean()
    return {"mean":float(arr.mean()),"ci_low":float(np.quantile(reps,.025)),"ci_high":float(np.quantile(reps,.975)),"n_systems":len(arr)}

ridge_by_country=country_df.groupby("country",as_index=False)["rho"].mean()
assert len(ridge_by_country)==63
print("Ridge system-equal rho, 95% CI:",bootstrap_system_mean(ridge_by_country["rho"]))
if comparison_country_df is not None:
    by_country=comparison_country_df.groupby("country",as_index=False)[["ridge_rho","crest_rho","delta_crest_minus_ridge"]].mean()
    assert len(by_country)==63
    report={key:bootstrap_system_mean(by_country[key],seed=BOOTSTRAP_SEED) for key in ["ridge_rho","crest_rho","delta_crest_minus_ridge"]}
    with open(BASELINE_DIR/"crest_vs_ridge_system_bootstrap.json","w",encoding="utf-8") as f: json.dump(report,f,indent=2)
    print("\nPaired CReST minus Ridge delta rho and 95% system bootstrap CI:")
    print(report["delta_crest_minus_ridge"])
    print("CReST mean rho here should approximately match published 0.515; check if not.")

Ridge system-equal rho, 95% CI: {'mean': 0.5028372029441531, 'ci_low': 0.5158062704747947, 'ci_high': 0.549017337728777, 'n_systems': 63}

Paired CReST minus Ridge delta rho and 95% system bootstrap CI:
{'mean': -0.017478989395457795, 'ci_low': -0.02753705438238195, 'ci_high': -0.00965641719772535, 'n_systems': 63}
CReST mean rho here should approximately match published 0.515; check if not.


## Interpretation and publication checklist

1. **Do not label this result an architectural superiority test.** This baseline replaces learned `z_general` with summary features but does not use the same representation calibration. Even if CReST wins, the difference could reflect feature engineering, regularisation, pretraining, or representation architecture.
2. **Avoid retrospective model selection.** Ridge α = 1 was fixed, not tuned to the 63 held-out systems. This new baseline was added post-hoc and should be reported as such.
3. **Avoid trial-and-error use of held-out systems.** Do not repeatedly change the feature set after seeing TEST results. If extending to LightGBM, use a separately specified development protocol.
4. **Matched data:** A/B items, held-out systems, target credit mapping, eligible students, and evaluation are identical by construction and audited against CReST `.npz` files.
5. **Next:** To compare the main *domain-relative* claim in Section 6.7, we must inspect/use Notebook **09** (plausible-value-conditioned targets) and run a separate source-only general vs general-plus-domain Ridge analysis. **Do not infer those Δρ values from this general transfer notebook.**
6. **Files saved:** `results/ridge_summary_baseline/` (14 pupil-level prediction NPZs, fold and system CSVs, feature registry, and optionally paired system CSV/bootstrap JSON).